Academic Integrity and Learning Statement

By submitting my work, I confirm that:

1. The code, analysis, and documentation in this notebook are my own work and reflect my own understanding.
2. I am prepared to explain all code and analysis included in this submission.

If I used assistance (e.g., AI tools, tutors, or other resources), I have:

- Clearly documented where and how external tools or resources were used in my solution.
- Included a copy of the interaction (e.g., AI conversation or tutoring notes) in an appendix.

I acknowledge that:

- I may be asked to explain any part of my code or analysis during evaluation.
- Misrepresenting assisted work as my own constitutes academic dishonesty and undermines my learning.

In [ ]:
# Import core python libraries
import numpy as np
import os, platform, multiprocessing
import pandas as pd

import torch

from sklearn.base import clone
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression

from src.models.trainer import create_submission
# Import custom python files
from src.utils import logging as prjlogger
from src.config import settings as prjconfigs
from src.visualisation import plots as prjplots
from src.data import loader as dloader
from src.features import engineering as prjfengg
from src.models import trainer as prjtrain
from src.utils.io import get_current_timestamp

In [ ]:
# Enable auto-reload extension
%load_ext autoreload
# Automatically reload all modules before executing code
%autoreload 2
%reload_ext autoreload

In [ ]:
# Check software specs
dict_sw_version = {
    'python': os.popen('python --version').read().strip(),
    'host platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__
}

for key, value in dict_sw_version.items():
    print(f'{prjplots.beautify(key, 1)} version is: {prjplots.beautify(value)}')

In [ ]:
logger = prjlogger.get_logger()
logger.debug(f"Initialised logging for {prjconfigs.PROJECT_NAME} project.")

In [ ]:
# Check CPU cores
print(f'CPU cores available to use: {prjplots.beautify(str(multiprocessing.cpu_count()))}')

# Machine info
print(f"Processor: {prjplots.beautify(platform.processor())}")
print(f"Machine: {prjplots.beautify(platform.machine())}")

# GPU info
print("PyTorch MPS (Metal) Status:")
print(f"MPS available: {prjplots.beautify(torch.backends.mps.is_available())}")
print(f"MPS built: {prjplots.beautify(str(torch.backends.mps.is_built()))}")
print(f"CUDA available: {prjplots.beautify(str(torch.cuda.is_available()))}")

In [ ]:
# Loading train & test data (raw files)
df_raw_train = dloader.load_data(prjconfigs.TRAIN_FILE, True)
df_raw_test = dloader.load_data(prjconfigs.TEST_FILE, True)

In [ ]:
# Probing few samples
df_raw_train.sample(3)

In [ ]:
# Columns tagging
insignificant_cols = ['Order', 'PID']
target_col = 'SalePrice'
ignorable_cols = insignificant_cols + [target_col]

temporal_cols_name_pattern = ['Yr', 'Year']

# Assigning ordinal columns based on manual heuristics and data review
ordinal_cols = ['LotShape', 'Utilities', 'LandSlope', 'OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

In [ ]:
# Merge train and test data
df_raw_all, df_raw_target = dloader.merge_train_test_data(
    df_raw_train,
    df_raw_test,
    insignificant_cols,
    target_col
)

In [ ]:
# Split into train and test based on the is_train column
df_train = df_raw_all[df_raw_all['is_train'] == 1].iloc[:, :-1]
df_test = df_raw_all[df_raw_all['is_train'] == 0].iloc[:, :-1]

In [ ]:
# Classify columns
feature_categories = prjfengg.classify_columns(
    df=df_train,
    n_cat_threshold=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL,
    threshold_type=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS,
    cols_to_ignore=ignorable_cols,
    temporal_cols_name_pattern=temporal_cols_name_pattern,
    ordinal_cols=ordinal_cols
)

In [ ]:
# Fetch column categories
(cols_num_continuous, n_cols_num_continuous, cols_num_discrete, n_cols_num_discrete,
 cols_cat_nominal, n_cols_cat_nominal, cols_cat_ordinal, n_cols_cat_ordinal,
 cols_object, n_cols_object, cols_temporal, n_cols_temporal,
 cols_binary, n_cols_binary, cols_low_cardinality, n_cols_low_cardinality) = prjfengg.get_cols_as_tuple(feature_categories)

In [ ]:
n_total_col_before_feat_split = df_train.shape[1]
n_total_col_after_feat_split = n_cols_num_continuous + n_cols_num_discrete + n_cols_cat_nominal + n_cols_cat_ordinal + n_cols_object + n_cols_temporal + n_cols_binary

print(f"-"*72)
print(
    f"Total raw columns      = {prjplots.beautify(str(len(df_train.columns)))}\n"
    f"Numerical Continuous   = {prjplots.beautify(n_cols_num_continuous)}\n"
    f"Numerical Discrete     = {prjplots.beautify(n_cols_num_discrete)}\n"
    f"Categorical Nominal    = {prjplots.beautify(n_cols_cat_nominal)}\n"
    f"Categorical Ordinal    = {prjplots.beautify(n_cols_cat_ordinal)}\n"
    f"Object/String          = {prjplots.beautify(n_cols_object)}\n"
    f"Temporal               = {prjplots.beautify(n_cols_temporal)}\n"
    f"Binary                 = {prjplots.beautify(n_cols_binary)}\n"
    f"Total feature columns  = {prjplots.beautify(n_total_col_after_feat_split)}"
)

has_inconsistency = n_total_col_before_feat_split != n_total_col_after_feat_split
inconsistency_val = prjplots.beautify("True", 3) if has_inconsistency else prjplots.beautify("False", 1)
print("-" * 72)
print(f"Any inconsistencies detected?[True/False] = {inconsistency_val}")
print(f'-'*72)

## Task 1: EDA and Data cleaning

In [ ]:
# Calculate the number of NaN values for each column
nan_counts = df_train.isna().sum()

# Filter only columns that have NaN values and sort by the number of NaNs
cols_with_nans = nan_counts[nan_counts > 0].index.tolist()
print(f"Columns with NaNs: = {prjplots.beautify(str(len(cols_with_nans)))}/{prjplots.beautify(n_total_col_before_feat_split)}")
print(f"And they are: {cols_with_nans}")

In [ ]:
# Create a temp DF to capture column cardinality info
tmp_df_cardinality = prjfengg.get_cardinality_df(df_train)
tmp_df_cardinality.sample(3)

### Task 1.1: Exploring missing data

In [ ]:
# Plot cardinality distribution of features in the dataset
prjplots.plot_cardinality(tmp_df_cardinality, prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL, threshold_used=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS, type_of_cols='all', figsize=(20, 6))

Insights & Observations:
Looking at the above plot (two facets: missing data percentage, and unique-value percentage), the following insights can be gathered.
1. Columns (on the left of the plot) like PoolQC, MiscFeature, Alley and Fence aren't truly missing. NaN in this context implies the property has no pool/alley/fence. And hence we may be better off by imputing the NaNs as a category
   "None", and not have it fill with mode. As filling with mode would incorrectly imply, most houses have a mediocre pool. A similar observation can be made for FireplaceQu.
2. There are a few column clusters exhibiting similar missingness like: Garage column cluster (GarageYrBlt, GarageFinish, GarageType, GarageQual, GarageCond) and Basement column cluster (BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1/2).
3. High unique_pct dashes for columns like: LotArea, GrLivArea, 1stFlrSF, TotalBsmtSF, MSSubClass: near-unique per row, which is no surprise as these are measurements. And hence justify StandardScaler.
4. Near-zero unique_pct (imply quasi-constant or binary features): Street, CentralAir, Utilities - dashes are essentially at 0, meaning only 2–3 unique values across 2430 rows
      - Utilities seem to be a near-constant in this dataset: ~99.9% of houses are "AllPub". It carries almost no predictive information and is a candidate for dropping.
      - Condition2 similarly nearly all rows are "Norm".

In [ ]:
# Dive into cols having NaNs only
prjplots.plot_cardinality(tmp_df_cardinality[tmp_df_cardinality['col_name'].isin(cols_with_nans)], prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL, threshold_used=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS, type_of_cols="NaN", figsize=(10, 6))

In [ ]:
# Creating a copy of the raw data to impute missing values for plotting purposes only (as NaNs are not plotted)
tmp_df_imputed_for_plots = df_train.copy()
tmp_df_imputed_for_plots[cols_num_continuous] = tmp_df_imputed_for_plots[cols_num_continuous].fillna(0)
most_frequent = tmp_df_imputed_for_plots[cols_num_discrete].mode().iloc[0]
tmp_df_imputed_for_plots[cols_num_discrete] = tmp_df_imputed_for_plots[cols_num_discrete].fillna(most_frequent)

In [ ]:
# Plot correlation between numerical features and target variable
correlation_plot = prjplots.plot_correlation_with_target(
    pd.concat([tmp_df_imputed_for_plots[cols_num_continuous], df_raw_target], axis=1),
    target_col
)

Insights & Observations:
Looking at the above plot (pearson correlation), the following insights can be gathered.
1. The top 5 features all measure usable square footage or capacity. Implying biggest (+ve corr) is Size-related attributes.
2. Few columns are multicollinear (e.g., GarageCars and GarageArea). This could be due to the nature of the data where larger garages tend to have more cars. Feeding both into a linear model (Lasso) may create redundancy, and better to drop one of them (for Lasso only).
3. Other few attributes like MasVnrArea & BsmtFinSF1 have medium signal and indicative of build quality. These could be important for predicting house prices.
4. EnclosedPorch has the only tangible -ve correlation with SalePrice. Indicating older, lower-value homes reflective of age and low-quality finish.

Also, important to consider that we are using corrwith() which is Pearson (linear). And hence several of the relationships which are likely non-linear in reality may not show up clearly in this chart. So will try to probe correlations further using alternative approaches.

In [ ]:
# Displays multicollinearity between the below features (as hypothesised under insight#2 above)
df_train[['GarageCars','GarageArea']].corr()

In [ ]:
# Plotting spearman correlation
sp_corr_plot = prjplots.plot_spearman_correlation_with_target(
    pd.concat([tmp_df_imputed_for_plots[cols_num_continuous], df_raw_target], axis=1),
    target_col
)

In [ ]:
# Plotting mutual information
mi_corr_plot = prjplots.plot_mutual_information_with_target(
    pd.concat([tmp_df_imputed_for_plots[cols_num_continuous], df_raw_target], axis=1),
    target_col
)

In [ ]:
# Plotting all correlations side-by-side to compare & contrast
corr_comparison_plot = prjplots.plot_feature_relevance_comparison(
  pd.concat([tmp_df_imputed_for_plots[cols_num_continuous], df_raw_target], axis=1),
  target_col
)

In [ ]:
prjplots.plot_numerical_distribution(tmp_df_imputed_for_plots, cols_num_continuous)

Insights & Observations:
Looking at the above plots (distribution), the following insights can be gathered.
1. 3SsnPorch, ScreenPorch, MiscVal, LowQualFinSF, BsmtFinSF2, EnclosedPorch: overwhelmingly zero (>85-95% of records). These features carry very little signal and may need transformation or dropping.
2. LotArea, MasVnrArea, GrLivArea, TotalBsmtSF, 1stFlrSF, BsmtFinSF1: having distributions with heavy right tails. Extreme outliers visible in boxplots (e.g., LotArea goes to ~200k vs. median ~10k). These benefit from log transformation before modeling.
3. GarageCars, BsmtFullBath, BsmtHalfBath — should be treated as ordinal/categorical rather than continuous. GarageCars distributed (0–4), BsmtFullBath mostly 0–1.
4. 2ndFlrSF, WoodDeckSF, OpenPorchSF, BsmtUnfSF: zero means the feature is absent (no 2nd floor, no deck, etc.). Creating a binary presence indicator alongside the continuous value could help models.

In [ ]:
prjplots.plot_categorical_distribution(tmp_df_imputed_for_plots, cols_cat_nominal)

Insights & Observations:
Looking at the above plots (distribution), the following insights can be gathered.
1. DOMINANT CATEGORIES: Heating, RoofMatl, Condition2, PavedDrive, essentially zero-variance. These add almost no discriminative signal and are candidates for dropping.
2. IMBALANCED CATEGORIES: LandContour, LotConfig, Condition1, RoofStyle, SaleCondition, Functional, SaleType, have one category with 75-85% records and may need grouping to avoid sparse one-hot columns.
3. HIGH CARDINALITY: Exterior1st and Exterior2nd have ~15 categories each, with VinylSd dominating. These may need rare-category consolidation before encoding.

In [ ]:
prjplots.plot_categorical_distribution(tmp_df_imputed_for_plots, cols_cat_ordinal)

Insights & Observations:
Looking at the above plots (distribution), the following insights can be gathered.
1. NEAR-CONSTANT CATEGORIES: Utilities, LandSlope, GarageQual, GarageCond, have virtually zero variance, minimal predictive value.
2. NICE DISTRIBUTION: OverallQual, BsmtFinType1, LotShape, BsmtQual, reasonable spread across categories, likely to carry genuine signal.
3. MIXED SIGNAL: BsmtExposure, BsmtFinType2, FireplaceQu: “absence" is the dominant state. These features are currently using a single category to represent both "feature is absent" and "feature is present but low quality" which are two fundamentally different states. This may mislead the model into treating absence as just another quality level. We might need to have a binary presence flag plus a quality ordinal applied only where the feature exists.

In [ ]:
prjplots.plot_relationship_to_target(pd.concat([tmp_df_imputed_for_plots[cols_num_discrete], df_raw_target], axis=1), cols_num_discrete, target_col)

Insights & Observations:
Looking at the above plot (relation of target variable to numerical features), the following insights can be gathered.
1. FullBath, TotRmsAbvGrd, and Fireplaces are the strongest predictors here (monotonic). And MoSold and PoolArea seemingly carry no signal and can be dropped.

In [ ]:
prjplots.plot_relationship_to_target(pd.concat([tmp_df_imputed_for_plots[cols_num_discrete], df_raw_target], axis=1), cols_num_discrete, target_col, trend_type='median')

Insights & Observations:

The median overlay sharpens the insights further. FullBath and Fireplaces are clean positives; BedroomAbvGr, HalfBath, TotRmsAbvGrd have non-linear behaviour; PoolArea and MoSold are confirmed noise.

In [ ]:
# Split train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    df_train,
    df_raw_target,
    test_size=prjconfigs.VALIDATION_SIZE,
    random_state=prjconfigs.RANDOM_STATE
)

In [ ]:
rows = [
  ("Raw TRAIN data variable:",        "df_raw_train",  df_raw_train.shape),
  ("Raw TEST data variable:",         "df_raw_test",   df_raw_test.shape),
  ("Raw ALL data variable:",          "df_raw_all",    df_raw_all.shape),
  ("Raw TARGET data variable:",       "df_raw_target", df_raw_target.shape),
  ("TRAIN data variable:",            "df_train",      df_train.shape),
  ("TEST data variable:",             "df_test",       df_test.shape),
  ("Validation TRAIN data variable:", "X_train",       X_train.shape),
  ("Validation VAL data variable:",   "X_val",         X_val.shape),
]

for label, varname, shape in rows:
  print(f"{label:<32}{(prjplots.beautify(varname.ljust(16)))} having shape = {prjplots.beautify(shape, 1)}")

In [ ]:
# Logically grouping columns as needed by the feature engineering transformer pipeline below (for intermediate and complex models)
num_columns = cols_num_continuous
cat_columns = cols_cat_nominal + cols_num_discrete + cols_binary + cols_object # included object as it is low cardinality
cat_ordinal_columns = cols_cat_ordinal
tempo_columns = cols_temporal

In [ ]:
len(num_columns), len(cat_columns), len(cat_ordinal_columns), len(tempo_columns)

## Baseline

- Implementing a baseline model aiming to predict house prices using a mean-price approach (agnostic of any features).
- The method has CV and score (dict object) implemented to have consistency with other methods/approaches which are explored later.

In [ ]:
# Without log transforming the target
baseline_results = prjtrain.run_baseline(y_train, y_val, log_target=False)

In [ ]:
# With log-transforming the target
baseline_results_logt = prjtrain.run_baseline(np.log(y_train), np.log(y_val), log_target=True)

In [ ]:
# Displaying CV results for both baseline approaches (log-transformed target vs non-transformed target).
print(
    f"CV Metrics for Baseline Model"
    f"\n{'-'*50}"
    f"\n{'Baseline (non-log variant) MAE':<31}: ${baseline_results['cv_mae_mean']:,.0f} ± ${baseline_results['cv_mae_std']:,.0f}"
    f"\n{'Baseline (log variant) MAE':<31}: ${baseline_results_logt['cv_mae_mean']:,.0f} ± ${baseline_results_logt['cv_mae_std']:,.0f}"
    f"\n{'-'*50}"
)

#### Insights & Observation:
- Log-transforming the target modestly improves the baseline, even for just predicting the mean.
- It seems to be a signal that log-transforming will likely benefit actual models too.

## Simple Model

- Implementing a simple Linear Regression model by first choosing limited set of features (limited to 2) via SelectKBest approach.

In [ ]:
# Set the number of features to be picked for Simple Model implementation
n_features_to_pick = 2

In [ ]:
# Creating temporary DF for numeric columns only (raw columns, without any encodings and pre-processing, which will be done later)
X_train_for_simple_model = X_train[cols_num_continuous].copy()
X_val_for_simple_model = X_val[cols_num_continuous].copy()
X_train_for_simple_model[cols_num_continuous] = X_train_for_simple_model[cols_num_continuous].fillna(0)
X_val_for_simple_model[cols_num_continuous] = X_val_for_simple_model[cols_num_continuous].fillna(0)

In [ ]:
# Without log-transforming the target
simple_model_results = prjtrain.run_simple_model(X_train_for_simple_model, y_train, X_val_for_simple_model, y_val, k=n_features_to_pick, log_target=False)

In [ ]:
# Without log-transforming the target
simple_model_results_logt = prjtrain.run_simple_model(X_train_for_simple_model, np.log(y_train), X_val_for_simple_model, np.log(y_val), k=n_features_to_pick, log_target=True)

In [ ]:
simple_model_features = simple_model_results['selected_features']

In [ ]:
# Displaying CV results for both simple model approaches (log-transformed target vs non-transformed target).
print(
    f"CV Metrics for Simple Model"
    f"\n{'-'*50}"
    f"\n{'Simple Model (non-log variant) MAE':<31}: ${simple_model_results['cv_mae_mean']:,.0f} ± ${simple_model_results['cv_mae_std']:,.0f}"
    f"\n{'Simple Model (log variant) MAE':<31}: ${simple_model_results_logt['cv_mae_mean']:,.0f} ± ${simple_model_results_logt['cv_mae_std']:,.0f}"
    f"\n{'-'*50}"
    f"\n{'List of features picked are':<31}: {simple_model_results['selected_features']}"
)

#### Insights & Observation:
- Simple Model beats baseline convincingly
- TotalBsmtSF and GrLivArea (total square footage signals) carry real predictive signal. This is expected (based on our EDA analysis).
- Interestingly (and in contrast to Baseline), the log variant is both worse on average and far more unstable (std of $14K vs $3K).
- Most likely because, with only two features, the log transformation distorts the linear relationship.
- Log-transforming the target but feeding in raw (non-log) feature values is, in essence, asking LR to fit a non-linear relationship with a linear model.
- It can't do that well, and hence shows up as having worse MAE and high std.

#### Pre-Modelling Set up

In [ ]:
experiment_id = prjtrain.get_or_create_experiment(prjconfigs.PROJECT_NAME)
print(f"MLflow experiment_id: {experiment_id}")

In [ ]:
# All MLFlow runs will be queryable via: mlflow ui as set below
prjtrain.set_mlflow_uri()

# NTS: This is the backend storage location — where run data, metrics, params, and artefacts are physically written to.
# Set this once before anything else.

In [ ]:
# NTS: always set the URI first, then the experiment. If the experiment is set before the URI,
# MLflow creates it in the wrong location (the default ./mlruns folder)
prjtrain.set_mlflow_experiment(prjconfigs.PROJECT_NAME)

## Intermediate Model

- TBC

In [ ]:
# Set the number of features to be picked for Intermediate Model implementation
n_features_to_pick = 15

In [ ]:
# Creating temporary DF of raw columns, without any encodings and pre-processing, which will be done later
mi_inter_cols = (
  cols_num_continuous + cols_num_discrete + cols_temporal +
  cols_cat_ordinal + cols_cat_nominal + cols_object
)

X_train_for_intermediate_model = X_train[mi_inter_cols].copy()

In [ ]:
# Impute missing values — median for numeric
for col in cols_num_continuous + cols_num_discrete + cols_temporal:
  X_train_for_intermediate_model[col] = X_train_for_intermediate_model[col].fillna(X_train_for_intermediate_model[col].median())
# Impute missing values — mode for categorical
for col in cols_cat_ordinal + cols_cat_nominal + cols_object:
  X_train_for_intermediate_model[col] = X_train_for_intermediate_model[col].fillna(X_train_for_intermediate_model[col].mode()[0])

In [ ]:
# Label-encode all non-numeric columns (for MI estimation order doesn't matter, so Label-encoding should be fine)
cols_to_encode = cols_cat_ordinal + cols_cat_nominal + cols_object
for col in cols_to_encode:
  le = LabelEncoder()
  X_train_for_intermediate_model[col] = le.fit_transform(X_train_for_intermediate_model[col].astype(str))

In [ ]:
# Creating mask for discrete features, which helps MI estimation
discrete_cols = set(
  cols_num_discrete + cols_cat_ordinal + cols_cat_nominal + cols_object
)
discrete_mask = [col in discrete_cols for col in mi_inter_cols]

In [ ]:
mi_scores = mutual_info_regression(
  X_train_for_intermediate_model, y_train,
  discrete_features=discrete_mask,
  random_state=prjconfigs.RANDOM_STATE
)
mi_ranking = (
  pd.Series(mi_scores, index=mi_inter_cols).round(3)
  .sort_values(ascending=False)
)
print(mi_ranking.to_string())

In [ ]:
top_k_features = mi_ranking.head(n_features_to_pick).index.tolist()
print(f"\nTop {n_features_to_pick} features selected:\n{top_k_features}")

In [ ]:
pproc_pipe_intermediate = prjfengg.create_pproc_pipeline(
  cols_num      = [c for c in num_columns       if c in top_k_features],
  cols_cat      = [c for c in cat_columns       if c in top_k_features],
  cols_ord_cat  = [c for c in cat_ordinal_columns if c in top_k_features],
  cols_temporal = [c for c in tempo_columns     if c in top_k_features],
)

In [ ]:
# One run_name per model — version + timestamp keeps them unique across re-runs
run_name_lasso = f"lasso_v{prjconfigs.MODEL_RUN_VERSION}_{get_current_timestamp()}"

artefact_path_lasso = f"model_lasso_v{prjconfigs.MODEL_RUN_VERSION}"

print(f"Lasso run: {run_name_lasso}")

In [ ]:
study_lasso, pipe_lasso, metrics_lasso = prjtrain.tune_lasso(
    X_train         =X_train[top_k_features],
    y_train         =y_train,
    X_val           =X_val[top_k_features],
    y_val           =y_val,
    pproc_pipeline  =pproc_pipe_intermediate,
    experiment_id   =experiment_id,
    run_name        =run_name_lasso,
    artefact_path   =artefact_path_lasso,
    num_trials      =prjconfigs.OPTUNA_TRIAL_COUNT
)

In [ ]:
study_lasso_logt, pipe_lasso_logt, metrics_lasso_logt = prjtrain.tune_lasso(
    X_train         =X_train[top_k_features],
    y_train         =np.log(y_train),
    X_val           =X_val[top_k_features],
    y_val           =np.log(y_val),
    pproc_pipeline  =pproc_pipe_intermediate,
    experiment_id   =experiment_id,
    run_name        =run_name_lasso,
    artefact_path   =artefact_path_lasso,
    num_trials      =prjconfigs.OPTUNA_TRIAL_COUNT,
    log_target      =True
)

In [ ]:
# Displaying CV results for both intermediate model approaches (log-transformed target vs non-transformed target).
print(
    f"CV Metrics for Intermediate Model"
    f"\n{'-'*50}"
    f"\n{'Intermediate Model (non-log variant) MAE':<43}: ${metrics_lasso['cv_mae_mean']:,.0f} ± ${metrics_lasso['cv_mae_std']:,.0f}"
    f"\n{'Intermediate Model (log variant) MAE':<43}: ${metrics_lasso_logt['cv_mae_mean']:,.0f} ± ${metrics_lasso_logt['cv_mae_std']:,.0f}"
    f"\n{'-'*50}"
    f"\n{'List of features picked are':<43}: {top_k_features}"
)

#### Insights & Observation:
- Each progression of the modelling approach cuts the error by roughly a third. The gains are consistent.
- Decreasing std (progressively from: baseline > simple > intermediate) implies the model is becoming more consistent across folds and less sensitive to which houses end up in which fold.

## Complex Model

- TBC

In [ ]:
# Probing the unique values of each categorical column to be used for ordinal categorisation later
for col in cols_cat_ordinal:
  unique_vals = sorted(df_train[col].dropna().unique().tolist())
  print(f"{col:<15} {unique_vals}")

In [ ]:
# Initialising the preprocessing pipeline
pproc_pipe = prjfengg.create_pproc_pipeline(num_columns, cat_columns, cat_ordinal_columns, tempo_columns)

In [ ]:
# Display an interactive visual summary of the preprocessing pipeline
pproc_pipe

In [ ]:
# Display columns passed through unprocessed
assigned = set(num_columns + cat_columns + cat_ordinal_columns + tempo_columns)
remainder_cols = [c for c in X_train.columns if c not in assigned]
print(f"Remainder cols ({len(remainder_cols)}): {remainder_cols}")

_fit_ learns the transformation parameters from the data — for example, the imputer learns the median/most-frequent value of each column, the scaler learns the mean and std, the ordinal encoder learns the category mappings. These learned values are stored inside the transformer objects.

_transform_ then applies those learned parameters to any data you pass in — it doesn't learn anything new, just applies what was learned during fit.

In [ ]:
# Fit the preprocessing pipeline on training data and transform it
# fit only on training data — so the pipeline learns statistics
# (mean, std, most frequent values, etc.) exclusively from training data
X_train_transformed = pproc_pipe.fit_transform(X_train)

In [ ]:
# Print feature expansion mapping for each transformer branch
prjfengg.print_feature_expansion(pproc_pipe)

In [ ]:
# Transform validation data using the already-fitted pipeline
# transform validation/test data using those same training-derived statistics —
# this prevents data leakage, where information from val/test data
# influences how the training data is preprocessed
X_val_transformed = pproc_pipe.transform(X_val)

In [ ]:
y_train_transformed = y_train.to_numpy()
y_val_transformed = y_val.to_numpy()

In [ ]:
# Check for both NaN and None
has_nulls_or_nans = pd.isna(X_train_transformed).any()
print(f"Transformed train data contains null or NaN values: {has_nulls_or_nans}")

In [ ]:
# One run_name per model — version + timestamp keeps them unique across re-runs
run_name_xgb   = f"xgb_v{prjconfigs.MODEL_RUN_VERSION}_{get_current_timestamp()}"

artefact_path_xgb   = f"model_xgb_v{prjconfigs.MODEL_RUN_VERSION}"

print(f"XGB run  : {run_name_xgb}")

In [ ]:
# XGBoost variant takes pre-transformed numpy arrays (pipeline already applied above)
study_xgb, pipe_xgb, metrics_xgb = prjtrain.tune_xgb(
  X_train        = X_train,
  y_train        = y_train,
  X_val          = X_val,
  y_val          = y_val,
  pproc_pipeline = pproc_pipe,
  experiment_id  = experiment_id,
  run_name       = run_name_xgb,
  artefact_path  = artefact_path_xgb,
  num_trials     = prjconfigs.OPTUNA_TRIAL_COUNT
)

In [ ]:
# XGBoost variant takes pre-transformed numpy arrays (pipeline already applied above)
study_xgb_logt, pipe_xgb_logt, metrics_xgb_logt = prjtrain.tune_xgb(
    X_train        = X_train,
    y_train        = np.log(y_train),
    X_val          = X_val,
    y_val          = np.log(y_val),
    pproc_pipeline = pproc_pipe,
    experiment_id  = experiment_id,
    run_name       = run_name_xgb,
    artefact_path  = artefact_path_xgb,
    num_trials     = prjconfigs.OPTUNA_TRIAL_COUNT,
    log_target    = True
)

In [ ]:
# Displaying CV results for both complex model approaches (log-transformed target vs non-transformed target).
print(
    f"CV Metrics for Complex Model"
    f"\n{'-'*50}"
    f"\n{'Complex Model (non-log variant) MAE':<43}: ${metrics_xgb['cv_mae_mean']:,.0f} ± ${metrics_xgb['cv_mae_std']:,.0f}"
    f"\n{'Complex Model (log variant) MAE':<43}: ${metrics_xgb_logt['cv_mae_mean']:,.0f} ± ${metrics_xgb_logt['cv_mae_std']:,.0f}"
    f"\n{'-'*50}"
)

#### Insights & Observation:
- The MAE & std continues to fall, and both are now consistently lower.
- The log vs non-log gap has nearly vanished. This is a meaningful trend as XGBoost is expressive enough to model the non-linear relationship between features and
  log-price natively and doesn't need the target transformation to linearise the problem.
- Every tier is progressively better than the one before it. Each increase in complexity is justified by performance gains.
- Baseline: no learning, just mean prediction
- Simple: 2 features, linear model
- Intermediate: 15 features, regularised & tuned model
- Complex: full feature set, non-linear & tuned model

In [ ]:
prjplots.plot_model_comparison({
    'Baseline':     baseline_results,
    'Simple':       simple_model_results,
    'Intermediate': metrics_lasso,
    'Complex':      metrics_xgb,
}, plot_type='bar')

In [ ]:
prjplots.plot_model_comparison({
    'Baseline':     baseline_results,
    'Simple':       simple_model_results,
    'Intermediate': metrics_lasso,
    'Complex':      metrics_xgb,
}, plot_type='box')

In [ ]:
# models_trained = ("XGBoost", study_xgb), ("Lasso", study_lasso)
models_trained = ("XGBoost", study_xgb)

In [ ]:
for name, study in [models_trained]:
  print(f"\n{'='*60}")
  print(f"  {name} — Study Summary")
  print(f"{'='*60}")
  print(f"  Best CV RMSE   : {study.best_value:.4f}")
  print(f"  Best trial #   : {study.best_trial.number}")
  print(f"  Total trials   : {len(study.trials)}")
  print(f"  Best params    :")
  for k, v in study.best_params.items():
      print(f"    {k:<25} = {v}")

In [ ]:
# Rich summary of each study's best params
prjtrain.print_study_summary({"XGBoost": study_xgb})

In [ ]:
# Top-N trials ranked by CV RMSE
prjtrain.print_trials_summary(study_xgb, "XGBoost", top_n=10)

In [ ]:
# Cross-model comparison from MLflow (with overfit gap indicator)
prjtrain.print_runs_comparison(experiment_id)

In [ ]:
pipe_xgb.named_steps

In [ ]:
prjplots.plot_feature_importance(pipe_xgb, top_n=30)

## Submission (Predicting on Test data)

Before performing predictions, the final model (respectively) needs to be trained on train+val dataset combined. So the workflow at a high level looks like:
- Step 1: Evaluate fit on X_train
- Step 2: Score on X_val (compare models, pick winner)
- Step 3: Final fit on train+val
- Step 4: Predict on test data

We've already performed #1 & #2; and now will progress on #3 & #4.

In [ ]:
# Concatenate train and val datasets to create a full training dataset view (for final fit)
X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

# Fetch PID data for test dataset
pid = df_raw_test["PID"]

# Re-name test DF to align with the submission-related naming convention
X_test = df_test

#### Simple Model Submission

In [ ]:
# Perform the final fit afresh (before prediction/submission)
simple_model_final = LinearRegression()
simple_model_final.fit(X_train_full[simple_model_features].fillna(0), y_full)

In [ ]:
# Derive predictions on test dataset (for submission)
create_submission(simple_model_final, X_test[simple_model_features].fillna(0), pid, "predictions-simple-model.csv")

#### Intermediate Model Submission

In [ ]:
# Perform the final fit afresh (before prediction/submission) using the best model (with learned params)

# Cloning to create an unfitted pipeline copy but having the same / learned hyperparameters; this just resets the fitted state.
intermediate_model_final = clone(pipe_lasso)
# Fitting on a full set of train data rows (but only on selected feature columns)
intermediate_model_final.fit(X_train_full[top_k_features], y_train_full)

In [ ]:
# Derive predictions on test dataset (for submission)
create_submission(intermediate_model_final, X_test[top_k_features], pid, "predictions-intermediate-model.csv")

#### Complex Model Submission

In [ ]:
# Perform the final fit afresh (before prediction/submission) using the best model (with learned params)

# Cloning to create an unfitted pipeline copy but having the same / learned hyperparameters; this just resets the fitted state.
complex_model_final = clone(pipe_xgb)
complex_model_final.fit(X_train_full, y_train_full)

In [ ]:
# Derive predictions on test dataset (for submission)
create_submission(complex_model_final, X_test, pid, "predictions-complex-model.csv")